# Invoice information extraction

This notebook contains **three independent sections**:

1. **Part A — `invoice_clear/`** — same layout on every page. **`extract_fields(..., layout="clear")`** forces the invoice_clear template parser.
2. **Part B — `images/`** — sample across all templates. **`extract_fields(..., layout="generic")`**.
3. **Part C — `images/` Template 1 only** — every `Template1_Instance*` file (good for checking **invoice date + due date** together on a homogeneous subset).

Extracted fields: invoice number, invoice date, due date, issuer name, recipient name, total amount.

**Stale outputs:** Jupyter keeps the **last executed** cell results in the file. After you change extraction code, use **Kernel → Restart & Run All** (or run cells from the top in order) so tables and `results_*` previews match the current code—not an older run.


## 1) Imports and OCR setup

If Tesseract is not on `PATH`, set `TESSERACT_CMD` in the environment or install Tesseract (see `README.md`).


In [1]:
from __future__ import annotations

import csv
import os
import re
import shutil
from datetime import datetime
from pathlib import Path
from typing import Iterable, Optional

import pytesseract

from scan import preprocess


def configure_tesseract() -> str:
    """Find and validate a working tesseract executable on Windows."""
    candidates = [
        os.environ.get("TESSERACT_CMD"),
        shutil.which("tesseract"),
        r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        str(Path.home() / "AppData/Local/Programs/Tesseract-OCR/tesseract.exe"),
        str(Path.home() / "scoop/apps/tesseract/current/tesseract.exe"),
    ]

    checked = []
    for candidate in candidates:
        if not candidate:
            continue

        candidate_path = Path(candidate)
        checked.append(str(candidate_path))
        if candidate_path.exists():
            pytesseract.pytesseract.tesseract_cmd = str(candidate_path)
            try:
                _ = pytesseract.get_tesseract_version()
                return str(candidate_path)
            except Exception:
                continue

    checked_str = "\n - ".join(sorted(set(checked)))
    raise RuntimeError(
        "Tesseract OCR was not found. Install it and set TESSERACT_CMD, or add it to PATH.\n"
        f"Checked:\n - {checked_str}"
    )


TESSERACT_PATH = configure_tesseract()
print("Using Tesseract:", TESSERACT_PATH)

Using Tesseract: C:\Program Files\Tesseract-OCR\tesseract.EXE


## 2) Shared configuration

Project root and allowed file extensions. Adjust sample sizes in Part A / Part B cells; Part C uses all Template 1 files under `images/`.


In [2]:
PROJECT_ROOT = Path.cwd()
SUPPORTED_EXTS = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: c:\Users\rdjg2\OneDrive\Escritorio\ie\3\2nd semester\Statistical learning\group project\Document-Classifier


## 3) Extraction helpers (shared)

Regex, normalizers, **`extract_text_robust`**, and **`extract_fields(text, layout=...)`** — use **`layout="clear"`** for `invoice_clear/`, **`layout="generic"`** for `images/`; **`layout="auto"`** (default) uses a strict SUMMARY + `Invoice no:` + Seller/Client heuristic.


In [3]:
DATE_REGEX = re.compile(
    r"\b(?:\d{1,2}[\-/]\d{1,2}[\-/]\d{2,4}|\d{4}[\-/]\d{1,2}[\-/]\d{1,2}|"
    r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\.?\s+\d{1,2},?\s+\d{2,4})\b",
    flags=re.IGNORECASE,
)

DATE_YEAR_MIN = 1960
DATE_YEAR_MAX = 2030

AMOUNT_REGEX = re.compile(r"(?<!\d)(?:[$€£]?\s*)\d{1,3}(?:[.,\s]\d{3})*(?:[.,]\d{2})(?!\d)")

def _first_match(patterns: Iterable[str], text: str) -> Optional[str]:
    for p in patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            value = m.group(1).strip(" :-\t\n")
            if value:
                return value
    return None


def _normalize_amount(value: Optional[str]) -> Optional[str]:
    if not value:
        return None

    raw = value.strip()
    raw = re.sub(r"[^0-9,.-]", "", raw)
    if not raw:
        return None

    if raw.count(",") > 0 and raw.count(".") > 0:
        if raw.rfind(",") > raw.rfind("."):
            raw = raw.replace(".", "")
            raw = raw.replace(",", ".")
        else:
            raw = raw.replace(",", "")
    elif raw.count(",") > 0 and raw.count(".") == 0:
        parts = raw.split(",")
        if len(parts[-1]) == 2:
            raw = "".join(parts[:-1]).replace("-", "") + "." + parts[-1]
        else:
            raw = "".join(parts)

    raw = raw.strip("-")
    try:
        return f"{float(raw):.2f}"
    except ValueError:
        return None



In [4]:
# Improved OCR + extraction (v2) for noisy historical scans
import unicodedata
from datetime import timedelta
from PIL import Image, ImageOps


def _clean_ocr_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).replace("\x00", "")
    lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines()]
    cleaned = [ln for ln in lines if ln]
    return "\n".join(cleaned)


def _ocr_image_multi_pass(path: Path) -> str:
    img = Image.open(path)
    gray = ImageOps.grayscale(img)

    # Fast pass first (most documents); extra passes only for hard scans.
    fast_cfg = "--oem 3 --psm 6"
    alt_cfg = "--oem 3 --psm 11"

    outputs = []

    try:
        txt = _clean_ocr_text(pytesseract.image_to_string(gray, config=fast_cfg))
        if txt:
            outputs.append(txt)
    except Exception:
        pass

    # Low-contrast fallback for difficult scans.
    if not outputs or len(outputs[0]) < 120:
        try:
            enhanced = ImageOps.autocontrast(gray)
            txt = _clean_ocr_text(pytesseract.image_to_string(enhanced, config=alt_cfg))
            if txt:
                outputs.append(txt)
        except Exception:
            pass

    # Black/white (binarized) fallback for very noisy historical scans.
    # Try two thresholds because scans can have different background darkness.
    if not outputs or max(len(x) for x in outputs) < 140:
        for thr in (145, 175):
            try:
                bw = gray.point(lambda x, t=thr: 255 if x > t else 0)
                txt = _clean_ocr_text(pytesseract.image_to_string(bw, config=alt_cfg))
                if txt:
                    outputs.append(txt)
            except Exception:
                pass

    # Add baseline OCR from scan.preprocess to recover lines missed by tuned configs.
    try:
        base_txt = _clean_ocr_text(preprocess(path))
        if base_txt:
            outputs.append(base_txt)
    except Exception:
        pass

    if not outputs:
        raise RuntimeError(f"OCR failed for file: {path}")

    # Keep best OCR result by alphanumeric density and length.
    def _score(t: str) -> tuple[int, int]:
        alnum = sum(ch.isalnum() for ch in t)
        return (alnum, len(t))

    best = max(outputs, key=_score)

    # Add extra lines from alternate passes if available.
    merged_lines = []
    seen = set()
    for txt in outputs:
        for ln in txt.splitlines():
            key = ln.lower().strip()
            if key and key not in seen:
                seen.add(key)
                merged_lines.append(ln)
    return "\n".join(merged_lines) if merged_lines else best


def extract_text_robust(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}:
        return _ocr_image_multi_pass(path)
    return preprocess(path)


def _parse_date_candidate(value: str | None) -> str | None:
    if not value:
        return None

    val = re.sub(r"\s+", " ", value.strip()).replace(",", "")
    val = val.replace(".", "/")

    fmts = [
        "%d/%m/%Y", "%d/%m/%y", "%m/%d/%Y", "%m/%d/%y", "%Y/%m/%d",
        "%d-%m-%Y", "%d-%m-%y", "%m-%d-%Y", "%m-%d-%y", "%Y-%m-%d",
        "%d-%b-%Y", "%d-%b-%y", "%d/%b/%Y", "%d/%b/%y",
        "%b %d %Y", "%B %d %Y", "%b %d %y", "%B %d %y",
        "%b. %d %Y", "%B. %d %Y", "%b. %d %y", "%B. %d %y",
        "%d %b %Y", "%d %B %Y", "%d %b %y", "%d %B %y",
    ]
    for fmt in fmts:
        try:
            dt = datetime.strptime(val, fmt).date()
            if DATE_YEAR_MIN <= dt.year <= DATE_YEAR_MAX:
                return dt.isoformat()
        except ValueError:
            continue

    m = DATE_REGEX.search(val)
    if m:
        found = m.group(0).strip()
        if found.lower() == val.strip().lower():
            return None
        return _parse_date_candidate(found)
    return None


_INLINE_DATE_ANY = re.compile(
    r"\b\d{1,2}[./-]\s*\d{1,2}[./-]\s*\d{2,4}\b"
    r"|\b\d{1,2}\s*[-/]\s*[A-Za-z]{3,9}\s*[-/]\s*\d{2,4}\b"
    r"|\b[A-Za-z]{3,9}\.?\s+\d{1,2},?\s+\d{2,4}\b"
    r"|\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4}\b"
    r"|\b\d{4}[./-]\s*\d{1,2}[./-]\s*\d{1,2}\b",
    flags=re.IGNORECASE,
)


def _parse_any_date_in_chunk(chunk: str | None, *, prefer_last: bool = False) -> str | None:
    """Parse date(s) from a label capture; use prefer_last when the value tail is the real date (OCR noise left)."""
    if not chunk:
        return None
    chunk = chunk.strip()
    if (d := _parse_date_candidate(chunk)):
        return d
    found: list[str] = []
    for m in _INLINE_DATE_ANY.finditer(chunk):
        if (d := _parse_date_candidate(m.group(0))):
            found.append(d)
    if not found:
        return None
    if prefer_last or re.search(r"(?i)\bdue\b|pay(?:ment)?\s*by|remit|balance\s*due", chunk):
        return found[-1]
    return found[0]


def _likely_company_line(line: str) -> bool:
    low = line.lower()
    if len(line) < 6:
        return False
    if re.search(
        r"invoice|date|total|amount|page|phone|fax|www|http|remittance|estimate|"
        r"duplicate|triplicate|duns|received|routing|protective order",
        low,
    ):
        return False
    letters = sum(c.isalpha() for c in line)
    if letters < 10 or letters / max(len(line), 1) < 0.35:
        return False
    words = re.findall(r"[A-Za-z&'.-]+", line)
    if len(words) < 2:
        return False
    corp = re.search(
        r"\b(inc\.?|corp\.?|co\.?|llc|ltd|company|corporation|associates|advertising|tobacco)\b",
        low,
    )
    if corp:
        return True
    return len(line) >= 18 and letters >= 12


def _is_valid_invoice_number(value: str | None) -> bool:
    if not value:
        return False

    raw = value.strip()
    token = raw.upper().replace(" ", "")
    if len(token) < 4 or len(token) > 28:
        return False

    banned_sub = (
        "DAYSNET", "30DAY", "SHEET", "ADDITION", "CHARGES", "MISCELLANEOUS",
        "PHOTOGRAPHER", "COMMISSION", "ESTIMATE", "INSERTION", "NSERTION",
        "BILLING", "PLACEMENT", "PRODUCTION", "RECEIVED", "ROUTING",
    )
    if any(s in token for s in banned_sub):
        return False

    banned_exact = {
        "INVOICE", "VOICE", "OICE", "REPORT", "REPGRT", "ORIGINAL", "DUPLICATE",
        "TRIPLICATE", "TRFLICATE", "BALANCE", "SALE", "EXPORT", "DATE", "NUMBER",
        "NUMBERS", "NONE", "NANO", "NANOO", "OATE", "MUNBER", "PHOTOGRAPHERARTIST",
        "NET", "TOTAL", "GROSS", "PINRT",
    }
    if token in banned_exact:
        return False

    if re.search(r"[A-Z]{10,}", token) and sum(ch.isdigit() for ch in token) < 2:
        return False

    dnorm = re.sub(r"[^0-9A-Z]", "", token).replace("O", "0").replace("I", "1").replace("L", "1")
    if re.fullmatch(r"\d{10}", dnorm):
        return False
    if re.fullmatch(r"\d{3}[-.]?\d{3}[-.]?\d{4}", dnorm):
        return False

    norm_digits = re.sub(r"\D", "", token.replace("O", "0").replace("o", "0"))
    if (
        token.count("-") >= 2
        and 8 <= len(norm_digits) <= 11
        and sum(c.isalpha() for c in token) <= 2
    ):
        return False

    if re.fullmatch(r"\d+[\-/]\d+[\-/]\d+", token):
        return False
    if re.fullmatch(r"\d{5}(?:-\d{4})?", token):
        return False
    if token.count("/") >= 2 and sum(ch.isdigit() for ch in token) < 5:
        return False

    digit_count = sum(ch.isdigit() for ch in token)
    alpha_count = sum(ch.isalpha() for ch in token)

    if len(token) > 12 and alpha_count > 8 and digit_count < 4:
        return False

    if len(token) <= 5 and alpha_count >= 2 and digit_count <= 2:
        return False
    if alpha_count == 0 and digit_count < 5:
        return False
    if alpha_count > 0 and digit_count < 2:
        return False

    return True


def _extract_invoice_number(text: str, lines: list[str]) -> str | None:
    label_patterns = [
        r"(?:invoice|inv|factura)\s*(?:number|no\.?|#|num(?:ber)?|id)\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-/]{2,28})",
        r"(?:our|your|customer)\s+ref(?:erence)?\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-/]{2,28})",
        r"(?:job|order|document)\s*(?:number|no\.?|#)?\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-/]{2,28})",
        r"(?:stmt|statement)\s*(?:number|no\.?|#)?\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-/]{2,28})",
        r"(?:no\.?|#)\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-/]{3,28})",
    ]
    for p in label_patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            cand = m.group(1).strip(" -:")
            if _is_valid_invoice_number(cand):
                return cand

    skip_ln = re.compile(r"duns|phone|tel\.|fax|zip|postal", re.I)
    for ln in lines[:45]:
        if skip_ln.search(ln):
            continue
        for tok in re.findall(r"\b[A-Z0-9][A-Z0-9\-/]{3,27}\b", ln, flags=re.I):
            if _is_valid_invoice_number(tok):
                return tok
    return None


def _extract_party(lines: list[str], role: str) -> str | None:
    if role == "recipient":
        labels = ["bill to", "to", "sold to", "ship to", "customer", "client", "attn"]
    else:
        labels = ["from", "vendor", "seller", "issuer", "remit to", "company"]

    for i, ln in enumerate(lines):
        low = ln.lower().strip()
        for lab in labels:
            if re.search(rf"\b{re.escape(lab)}\b\s*:?", low):
                # inline value after colon
                if ":" in ln:
                    inline = ln.split(":", 1)[1].strip()
                    if _likely_company_line(inline):
                        return inline
                # or next meaningful line(s)
                for j in range(i + 1, min(i + 5, len(lines))):
                    cand = lines[j].strip(" :-\t")
                    if _likely_company_line(cand):
                        return cand

    def _party_score(ln: str) -> int:
        low = ln.lower()
        score = 0
        for kw in ("inc", "corp", "llc", "ltd", "tobacco", "advertising", "international", "company"):
            if kw in low:
                score += 4
        if "&" in ln:
            score += 2
        return score

    top_candidates = sorted(
        [ln for ln in lines[:28] if _likely_company_line(ln)],
        key=_party_score,
        reverse=True,
    )
    if role == "issuer" and top_candidates:
        return top_candidates[0]
    if role == "recipient" and len(top_candidates) >= 2:
        first = top_candidates[0]
        for cand in top_candidates[1:]:
            if cand[:32].lower() != first[:32].lower():
                return cand
        return top_candidates[1]
    if role == "recipient" and top_candidates:
        return top_candidates[0]
    return None


def _pick_total_from_amounts(amounts: list[str]) -> str | None:
    vals = sorted({float(a) for a in amounts if a}, reverse=True)
    if not vals:
        return None
    while len(vals) >= 2 and vals[0] > 25 * vals[1] and vals[1] >= 1:
        vals.pop(0)
    best = vals[0]
    if best >= 500_000 and len(vals) >= 2 and vals[1] < best / 10:
        best = vals[1]
    return f"{best:.2f}"


def _extract_total_amount(text: str, lines: list[str]) -> str | None:
    total_lines = [
        ln for ln in lines
        if re.search(
            r"grand\s*total|total\s*due|amount\s*due|balance\s*due|net\s*amount|"
            r"\btotals\b|\btotal\b",
            ln,
            re.IGNORECASE,
        )
    ]

    candidates: list[str] = []
    for ln in total_lines:
        for amt in AMOUNT_REGEX.findall(ln):
            norm = _normalize_amount(amt)
            if norm:
                candidates.append(norm)

    if candidates:
        picked = _pick_total_from_amounts(candidates)
        if picked:
            return picked

    for amt in AMOUNT_REGEX.findall(text):
        norm = _normalize_amount(amt)
        if norm:
            candidates.append(norm)

    if not candidates:
        return None

    filtered = [c for c in candidates if float(c) >= 5]
    pool = filtered if filtered else candidates
    return _pick_total_from_amounts(pool)


_DUE_LINE_PATTERNS = [
    re.compile(r"(?i)(?:due|[o0]ue|dne|cue)\s*(?:date|dae|data|dste)?\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bdue\s*date\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bdate\s*due\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bpayment\s*due\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\b(?:latest\s*)?payment\s*(?:by|before)\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bpay(?:ment)?\s*by\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bremit(?:tance)?\s*(?:by|before)?\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bbalance\s*due\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\b(?:deadline|good\s*until|must\s*be\s*paid)\s*[:.\-\s]+\s*(.+)$"),
    re.compile(r"(?i)\bdue\s*[:.\-\s]+\s*(.+)$"),
]


def _labeled_due_date_from_lines(lines: list[str], invoice_date: str | None) -> str | None:
    """Scan line-by-line for due-style labels (due may be before invoice on synthetic data)."""
    skip_start = re.compile(
        r"(?i)^(the|upon|receipt|immediately|cash|wire|ach|transfer|see\s+reverse|t\.?\s*b\.?\s*d\.?|asap)\b",
    )
    for ln in lines[:120]:
        s = ln.strip()
        if not s:
            continue
        for pat in _DUE_LINE_PATTERNS:
            m = pat.search(s)
            if not m:
                continue
            frag = m.group(1).strip()
            if not frag or skip_start.match(frag):
                continue
            pd = _parse_any_date_in_chunk(frag, prefer_last=True)
            if not pd:
                continue
            try:
                y = datetime.fromisoformat(pd).year
                if DATE_YEAR_MIN > y or y > DATE_YEAR_MAX:
                    continue
            except Exception:
                continue
            if invoice_date and pd == invoice_date:
                continue
            return pd
    return None


_DUEISH_LINE_HINT = re.compile(
    r"(?i)\b(due|payment\s*due|pay\s*by|remit(?:tance)?|balance\s*due|"
    r"latest\s*payment|deadline|good\s*until)\b",
)


def _due_dates_from_dueish_lines(lines: list[str], invoice_date: str | None) -> str | None:
    """Lines mentioning due/payment-by: collect dates; prefer one different from invoice_date when known."""
    for ln in lines[:150]:
        s = ln.strip()
        if not s or not _DUEISH_LINE_HINT.search(s):
            continue
        if re.search(r"(?i)\b(invoice|issue|issued)\s*date\b", s) and not re.search(r"(?i)\bdue\b", s):
            continue
        dates: list[str] = []
        for m in _INLINE_DATE_ANY.finditer(s):
            if (d := _parse_date_candidate(m.group(0))):
                dates.append(d)
        if not dates and (d := _parse_date_candidate(s)):
            dates.append(d)
        if not dates:
            continue
        if invoice_date:
            for pd in dates:
                if pd != invoice_date:
                    return pd
            return dates[-1]
        return dates[-1]
    return None


def _sanitize_date_pair(invoice_date: str | None, due_date: str | None) -> tuple[str | None, str | None]:
    def _valid_year(d: str | None) -> bool:
        if not d:
            return False
        try:
            y = datetime.fromisoformat(d).year
            return DATE_YEAR_MIN <= y <= DATE_YEAR_MAX
        except Exception:
            return False

    if not _valid_year(invoice_date):
        invoice_date = None
    if not _valid_year(due_date):
        due_date = None

    # Do not enforce due >= invoice: many templates (and OCR) use arbitrary or synthetic chronology.

    return invoice_date, due_date


def _extract_fields_generic(text: str) -> dict:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    invoice_number = _extract_invoice_number(text, lines)

    invoice_date_raw = _first_match([
        r"invoice\s*date\s*[:\-]?\s*([^\n]{3,45})",
        r"date\s*of\s*issue\s*[:\-]?\s*([^\n]{3,45})",
        r"(?:issue|issued)\s*date\s*[:\-]?\s*([^\n]{3,45})",
        r"(?im)^\s*Date\s*:\s*([^\n]{3,45})",
        r"(?m)^\s*Date\s*[:\-]\s*([0-9]{1,2}[-/][A-Za-z]{3,9}[-/][0-9]{2,4})",
        r"dated\s*[:\-]?\s*([^\n]{3,45})",
    ], text)

    due_date_raw = _first_match([
        r"(?im)^\s*Due\s+Date\s*:\s*([^\n;]{3,55})",
        r"(?i)\bdue\s+date\s*:\s*([^\n;]{3,55})",
        r"(?is)due\s*date\s*[:.\-]?\s*\n\s*([^\n;]{3,55})",
        r"due\s*date\s*[:.\-]?\s*([^\n;]{3,55})",
        r"date\s*due\s*[:.\-]?\s*([^\n;]{3,55})",
        r"payment\s*due\s*[:.\-]?\s*([^\n;]{3,55})",
        r"(?:latest\s*)?payment\s*(?:by|before)\s*[:.\-]?\s*([^\n;]{3,55})",
        r"pay(?:ment)?\s*by\s*[:.\-]?\s*([^\n;]{3,55})",
        r"remit(?:tance)?\s*(?:by|before)?\s*[:.\-]?\s*([^\n;]{3,55})",
        r"balance\s*due\s*[:.\-]?\s*([^\n;]{3,55})",
        r"(?:deadline|good\s*until)\s*[:.\-]?\s*([^\n;]{3,55})",
        r"(?m)^\s*due\s*[:.\-]\s*([^\n;]{3,55})",
        r"(?:payment\s*)?terms\s*[:.\-]?\s*net\s*(\d{1,3})\s*days?",
        r"\bnet\s*(\d{1,3})\s*days?\b",
    ], text)

    date_candidates = [d for d in DATE_REGEX.findall(text)]
    loose_date_candidates = re.findall(r"\b\d{1,2}\s*[./-]\s*\d{1,2}\s*[./-]\s*\d{2,4}\b", text)
    alpha_month_dates = re.findall(
        r"\b\d{1,2}\s*[-/]\s*[A-Za-z]{3,9}\s*[-/]\s*\d{2,4}\b", text
    )
    invoice_date = _parse_any_date_in_chunk(invoice_date_raw)
    due_date = _parse_any_date_in_chunk(due_date_raw, prefer_last=True)

    if not invoice_date and date_candidates:
        invoice_date = _parse_date_candidate(date_candidates[0])
    if not invoice_date and loose_date_candidates:
        invoice_date = _parse_date_candidate(loose_date_candidates[0])
    if not invoice_date and alpha_month_dates:
        invoice_date = _parse_date_candidate(alpha_month_dates[0])

    if not due_date and len(date_candidates) > 1:
        due_date = _parse_date_candidate(date_candidates[1])
    if not due_date and len(loose_date_candidates) > 1:
        due_date = _parse_date_candidate(loose_date_candidates[1])
    if not due_date and len(alpha_month_dates) > 1:
        due_date = _parse_date_candidate(alpha_month_dates[1])

    if invoice_date and not due_date:
        try:
            for dc in alpha_month_dates + loose_date_candidates + date_candidates:
                pd = _parse_date_candidate(dc)
                if pd and pd != invoice_date:
                    due_date = pd
                    break
        except Exception:
            pass

    if not due_date:
        due_date = _labeled_due_date_from_lines(lines, invoice_date)
    if not due_date:
        due_date = _due_dates_from_dueish_lines(lines, invoice_date)
    if not due_date and invoice_date:
        due_date = _labeled_due_date_from_lines(lines, None)
        if not due_date:
            due_date = _due_dates_from_dueish_lines(lines, None)

    # If terms were captured as number of days, estimate due date.
    if not due_date and due_date_raw and due_date_raw.isdigit() and invoice_date:
        try:
            inv_dt = datetime.fromisoformat(invoice_date)
            due_date = (inv_dt + timedelta(days=int(due_date_raw))).date().isoformat()
        except Exception:
            pass

    invoice_date, due_date = _sanitize_date_pair(invoice_date, due_date)

    issuer_name = _extract_party(lines, "issuer")
    recipient_name = _extract_party(lines, "recipient")
    total_amount = _extract_total_amount(text, lines)

    return {
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "due_date": due_date,
        "issuer_name": issuer_name,
        "recipient_name": recipient_name,
        "total_amount": total_amount,
    }


def _extract_block_between(text: str, start_label: str, stop_label: str) -> str | None:
    pattern = rf"{start_label}\s*:\s*(.*?){stop_label}\s*:"
    m = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if not m:
        return None
    block = m.group(1).strip()
    return block if block else None


def _looks_like_name(line: str) -> bool:
    low = line.lower().strip()
    if not low:
        return False
    if re.search(r"invoice|date|issue|tax\s*id|iban|items|summary|total|qty|gross|net|vat", low):
        return False
    if sum(ch.isdigit() for ch in line) > 2:
        return False
    words = re.findall(r"[A-Za-z&'.,-]+", line)
    return len(words) >= 1 and sum(ch.isalpha() for ch in line) >= 4


def _first_name_in_block(block: str | None) -> str | None:
    if not block:
        return None

    lines = [ln.strip(" :-\t") for ln in block.splitlines() if ln.strip()]
    for ln in lines:
        # Remove label echoes produced by OCR in side-by-side columns.
        cleaned = re.sub(r"\b(?:seller|client)\s*:\s*", "", ln, flags=re.IGNORECASE).strip()
        if cleaned.lower() in {"seller", "client"}:
            continue
        if _looks_like_name(cleaned):
            return cleaned
    return None


def _extract_clear_money_values(text: str) -> list[str]:
    strict_money = re.findall(r"\b\d{1,3}(?:\s\d{3})*(?:[.,]\d{2})\b", text)
    vals = [_normalize_amount(v) for v in strict_money]
    return [v for v in vals if v is not None and float(v) < 1_000_000]


def _split_possible_dual_names(line: str) -> tuple[str | None, str | None]:
    tokens = [t for t in line.split() if t]
    if len(tokens) < 4:
        return None, None

    corp_words = ("inc", "inc.", "ltd", "ltd.", "llc", "plc", "corp", "corporation")

    # Common case: second entity ends with legal suffix (e.g., "Becker Ltd").
    if tokens[-1].lower().strip(".,") in corp_words and len(tokens) >= 4:
        left = " ".join(tokens[:-2]).strip(" ,")
        right = " ".join(tokens[-2:]).strip(" ,")
        if _looks_like_name(left) and _looks_like_name(right):
            return left, right

    for i in range(2, len(tokens) - 1):
        left = " ".join(tokens[:i]).strip(" ,")
        right = " ".join(tokens[i:]).strip(" ,")
        if not (_looks_like_name(left) and _looks_like_name(right)):
            continue
        if any(w in right.lower() for w in corp_words) and not any(w in left.lower() for w in corp_words):
            return left, right

    # Generic split fallback for merged names without legal suffixes.
    for i in range(2, len(tokens) - 1):
        left = " ".join(tokens[:i]).strip(" ,")
        right = " ".join(tokens[i:]).strip(" ,")
        if _looks_like_name(left) and _looks_like_name(right):
            left_words = len(left.split())
            right_words = len(right.split())
            if 1 <= left_words <= 5 and 1 <= right_words <= 5:
                return left, right

    return None, None


def _extract_fields_clear_layout(text: str) -> dict:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    invoice_no_raw = _first_match([
        r"invoice\s*no\.?\s*:\s*([A-Z0-9\-/]{4,})",
        r"invoice\s*(?:number|#)\s*:\s*([A-Z0-9\-/]{4,})",
    ], text)
    invoice_number = invoice_no_raw if _is_valid_invoice_number(invoice_no_raw) else None

    invoice_date_raw = _first_match([
        r"date\s*of\s*issue\s*:\s*([0-9]{1,2}[\-/][0-9]{1,2}[\-/][0-9]{2,4})",
        r"date\s*of\s*issue\s*:\s*\n\s*([0-9]{1,2}[\-/][0-9]{1,2}[\-/][0-9]{2,4})",
    ], text)
    invoice_date = _parse_date_candidate(invoice_date_raw)

    # Fallback: in this template, the issue date may appear alone near SUMMARY.
    if not invoice_date:
        loose_dates = re.findall(r"\b\d{1,2}[\-/]\d{1,2}[\-/]\d{2,4}\b", text)
        parsed_dates = [d for d in (_parse_date_candidate(x) for x in loose_dates) if d]
        if parsed_dates:
            invoice_date = parsed_dates[0]

    # For this clear template, due date is only taken from explicit due/terms fields.
    due_date_raw = _first_match([
        r"due\s*date\s*:\s*([0-9]{1,2}[\-/][0-9]{1,2}[\-/][0-9]{2,4})",
        r"payment\s*due\s*:\s*([0-9]{1,2}[\-/][0-9]{1,2}[\-/][0-9]{2,4})",
        r"terms\s*:\s*net\s*(\d{1,3})\s*days",
    ], text)
    due_date = _parse_date_candidate(due_date_raw)
    if not due_date and due_date_raw and due_date_raw.isdigit() and invoice_date:
        try:
            inv_dt = datetime.fromisoformat(invoice_date)
            due_date = (inv_dt + timedelta(days=int(due_date_raw))).date().isoformat()
        except Exception:
            pass

    # Avoid copying invoice date into due date when not explicitly present.
    if due_date and invoice_date and due_date == invoice_date:
        due_date = None

    seller_block = _extract_block_between(text, "Seller", "Tax\s*Id")
    client_block = _extract_block_between(text, "Client", "Tax\s*Id")

    issuer_name = _first_name_in_block(seller_block)
    recipient_name = _first_name_in_block(client_block)

    # Handle side-by-side OCR case: "Seller: Client:" then one merged name line.
    if not recipient_name or (issuer_name and recipient_name and issuer_name == recipient_name):
        dual = re.search(r"seller\s*:\s*client\s*:?\s*\n\s*([^\n]{6,120})", text, flags=re.IGNORECASE)
        if dual:
            left, right = _split_possible_dual_names(dual.group(1))
            if left and right:
                issuer_name = left
                recipient_name = right

    # Recover missing party names from labeled lines when OCR collapses columns.
    if not issuer_name:
        issuer_name = _first_match([r"seller\s*:\s*([^\n:]{3,80})"], text)
    if not recipient_name:
        recipient_name = _first_match([r"client\s*:\s*([^\n:]{3,80})"], text)

    # Last fallback: split merged company name if both ended up equal.
    if issuer_name and recipient_name and issuer_name == recipient_name:
        left, right = _split_possible_dual_names(issuer_name)
        if left and right:
            issuer_name, recipient_name = left, right

    total_amount = None

    # Preferred source: explicit Total row in summary (last amount is gross total).
    total_line = next((ln for ln in lines if re.search(r"^total\b", ln, re.IGNORECASE)), None)
    if total_line:
        vals = _extract_clear_money_values(total_line)
        if vals:
            total_amount = vals[-1]

    # Fallback: scan summary section values and pick last strong candidate.
    if not total_amount:
        summary_block = text.split("SUMMARY", 1)[1] if "SUMMARY" in text else text
        summary_vals = _extract_clear_money_values(summary_block)
        if summary_vals:
            total_amount = summary_vals[-1]

    return {
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "due_date": due_date,
        "issuer_name": issuer_name,
        "recipient_name": recipient_name,
        "total_amount": total_amount,
    }


def _looks_like_invoice_clear_template(text: str) -> bool:
    """Stricter heuristic so AI invoices in images/ are not mistaken for invoice_clear."""
    if "SUMMARY" not in text.upper():
        return False
    if not re.search(r"(?i)\binvoice\s*no\.?\s*:", text):
        return False
    seller = _extract_block_between(text, "Seller", "Tax\s*Id")
    client = _extract_block_between(text, "Client", "Tax\s*Id")
    return bool(seller and client)


def extract_fields(text: str, layout: str = "auto") -> dict:
    """Extract invoice fields. Use layout='clear' for invoice_clear/, 'generic' for images/."""
    layout = (layout or "auto").strip().lower()
    if layout not in {"auto", "clear", "generic"}:
        layout = "auto"
    if layout == "clear" or (layout == "auto" and _looks_like_invoice_clear_template(text)):
        return _extract_fields_clear_layout(text)
    return _extract_fields_generic(text)


## Part A — Extract from `invoice_clear/`

Homogeneous template: `Invoice no:`, `Date of issue`, optional due date, **Seller** / **Client** blocks, `Total` gross line.


In [5]:
INVOICE_CLEAR_DIR = PROJECT_ROOT / "invoice_clear"
SAMPLE_SIZE_CLEAR = 10
invoice_clear_files = sorted(
    p for p in INVOICE_CLEAR_DIR.iterdir() if p.suffix.lower() in SUPPORTED_EXTS
)
invoice_clear_sample = invoice_clear_files[:SAMPLE_SIZE_CLEAR]
print(f"invoice_clear: {len(invoice_clear_files)} files, sample {len(invoice_clear_sample)}")
invoice_clear_sample[:3]


invoice_clear: 499 files, sample 10


[WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/invoice_clear/batch1-0001.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/invoice_clear/batch1-0002.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/invoice_clear/batch1-0003.jpg')]

### Part A — Run extraction


In [6]:
results_clear = []
errors_clear = []

for idx, path in enumerate(invoice_clear_sample, start=1):
    try:
        text = preprocess(path)
        row = extract_fields(text, layout="clear")
        row["file_name"] = path.name
        results_clear.append(row)
    except Exception as exc:
        errors_clear.append({"file_name": path.name, "error": str(exc)})
    if idx % 100 == 0:
        print(f"Part A: {idx}/{len(invoice_clear_sample)}...")

print(f"Part A processed: {len(results_clear)}, errors: {len(errors_clear)}")
results_clear[:3]


Part A processed: 10, errors: 0


[{'invoice_number': '51109338',
  'invoice_date': '2013-04-13',
  'due_date': None,
  'issuer_name': 'Andrews, Kirby and Valdez',
  'recipient_name': 'Becker Ltd',
  'total_amount': '6204.19',
  'file_name': 'batch1-0001.jpg'},
 {'invoice_number': '12847181',
  'invoice_date': '2012-03-03',
  'due_date': None,
  'issuer_name': 'Fitzpatrick and Sons',
  'recipient_name': 'Duncan PLC',
  'total_amount': '6860.45',
  'file_name': 'batch1-0002.jpg'},
 {'invoice_number': '19471831',
  'invoice_date': '2014-09-04',
  'due_date': None,
  'issuer_name': 'Palmer Ltd',
  'recipient_name': 'Rios, Oneill and Rowe',
  'total_amount': '44745.59',
  'file_name': 'batch1-0003.jpg'}]

### Part A — Save CSV


In [7]:
RESULTS_CLEAR_CSV = PROJECT_ROOT / "invoice_clear_extraction_results.csv"
ERRORS_CLEAR_CSV = PROJECT_ROOT / "invoice_clear_extraction_errors.csv"

fields = [
    "file_name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]
if results_clear:
    with RESULTS_CLEAR_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(results_clear)
    print("Saved:", RESULTS_CLEAR_CSV)
else:
    print("No Part A rows to save.")

if errors_clear:
    with ERRORS_CLEAR_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["file_name", "error"])
        w.writeheader()
        w.writerows(errors_clear)
    print("Saved:", ERRORS_CLEAR_CSV)


Saved: c:\Users\rdjg2\OneDrive\Escritorio\ie\3\2nd semester\Statistical learning\group project\Document-Classifier\invoice_clear_extraction_results.csv


### Part A — Field coverage (missing vs filled)

Each line is **missing count** and **filled count** out of the sample size (not “score out of 100”).


In [8]:
def _field_missing(v) -> bool:
    return v is None or (isinstance(v, str) and not v.strip())


req = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]
if results_clear:
    n = len(results_clear)
    for field in req:
        miss = sum(1 for r in results_clear if _field_missing(r.get(field)))
        ok = n - miss
        print(f"- {field}: {ok} filled, {miss} missing (n={n})")
else:
    print("No Part A results.")


- invoice_number: 10 filled, 0 missing (n=10)
- invoice_date: 10 filled, 0 missing (n=10)
- due_date: 0 filled, 10 missing (n=10)
- issuer_name: 10 filled, 0 missing (n=10)
- recipient_name: 10 filled, 0 missing (n=10)
- total_amount: 10 filled, 0 missing (n=10)


## Part B — Extract from `images/` (AI templates)

Files look like `Template12_Instance0.jpg`. This section uses **`preprocess` + `extract_fields(..., layout="generic")`**. Change **`AI_IMAGES_PER_TEMPLATE`** to take more or fewer examples per template ID.


In [9]:
from collections import defaultdict

IMAGES_DIR = PROJECT_ROOT / "images"
AI_IMAGES_PER_TEMPLATE = 2

by_tpl: dict[int, list] = defaultdict(list)
for p in sorted(IMAGES_DIR.iterdir()):
    if not p.is_file():
        continue
    if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}:
        continue
    m = re.match(r"Template(\d+)_Instance(\d+)", p.stem, re.I)
    if not m:
        continue
    by_tpl[int(m.group(1))].append((int(m.group(2)), p))

images_sample = []
for tid in sorted(by_tpl.keys()):
    paths = [pt for _, pt in sorted(by_tpl[tid], key=lambda x: x[0])][:AI_IMAGES_PER_TEMPLATE]
    images_sample.extend(paths)

print(f"images: {len(by_tpl)} templates, sample size {len(images_sample)}")
images_sample[:4]


images: 50 templates, sample size 100


[WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance0.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance1.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template2_Instance0.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template2_Instance1.jpg')]

### Part B — Run extraction


In [10]:
results_images = []
errors_images = []

for idx, path in enumerate(images_sample, start=1):
    try:
        text = preprocess(path)
        row = extract_fields(text, layout="generic")
        m = re.match(r"Template(\d+)_", path.name, re.I)
        row["template_id"] = int(m.group(1)) if m else None
        row["file_name"] = path.name
        results_images.append(row)
    except Exception as exc:
        errors_images.append({"file_name": path.name, "error": str(exc)})
    if idx % 25 == 0:
        print(f"Part B: {idx}/{len(images_sample)}...")

print(f"Part B processed: {len(results_images)}, errors: {len(errors_images)}")
results_images[:2]


Part B: 25/100...
Part B: 50/100...
Part B: 75/100...
Part B: 100/100...
Part B processed: 100, errors: 0


[{'invoice_number': '12345670',
  'invoice_date': '2008-03-20',
  'due_date': None,
  'issuer_name': 'Address:16424 Timothy Mission',
  'recipient_name': '16424 Timothy Mission',
  'total_amount': '734.33',
  'template_id': 1,
  'file_name': 'Template1_Instance0.jpg'},
 {'invoice_number': 'v-1999',
  'invoice_date': '1994-04-01',
  'due_date': None,
  'issuer_name': 'Address:16424 Timothy Mission',
  'recipient_name': 'Markville, AK 58294 US.',
  'total_amount': '92.46',
  'template_id': 1,
  'file_name': 'Template1_Instance1.jpg'}]

### Part B — Save CSV


In [11]:
RESULTS_IMAGES_CSV = PROJECT_ROOT / "images_ai_extraction_results.csv"
ERRORS_IMAGES_CSV = PROJECT_ROOT / "images_ai_extraction_errors.csv"

img_fields = [
    "template_id",
    "file_name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]
if results_images:
    with RESULTS_IMAGES_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=img_fields, extrasaction="ignore")
        w.writeheader()
        for r in results_images:
            w.writerow({k: r.get(k) for k in img_fields})
    print("Saved:", RESULTS_IMAGES_CSV)
else:
    print("No Part B rows to save.")

if errors_images:
    with ERRORS_IMAGES_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["file_name", "error"])
        w.writeheader()
        w.writerows(errors_images)
    print("Saved:", ERRORS_IMAGES_CSV)


Saved: c:\Users\rdjg2\OneDrive\Escritorio\ie\3\2nd semester\Statistical learning\group project\Document-Classifier\images_ai_extraction_results.csv


### Part B — Field coverage (missing vs filled)

Same as Part A: first number is **how many rows have the value**, second is **how many are empty**.


In [12]:
def _field_missing(v) -> bool:
    return v is None or (isinstance(v, str) and not v.strip())


req = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]
if results_images:
    n = len(results_images)
    for field in req:
        miss = sum(1 for r in results_images if _field_missing(r.get(field)))
        ok = n - miss
        print(f"- {field}: {ok} filled, {miss} missing (n={n})")
else:
    print("No Part B results.")


- invoice_number: 100 filled, 0 missing (n=100)
- invoice_date: 93 filled, 7 missing (n=100)
- due_date: 3 filled, 97 missing (n=100)
- issuer_name: 100 filled, 0 missing (n=100)
- recipient_name: 100 filled, 0 missing (n=100)
- total_amount: 100 filled, 0 missing (n=100)


## Part C — `images/` Template 1 only

Processes **every** file matching **`Template1_Instance*.`** (same extractor as Part B: **`preprocess` + `extract_fields(..., layout="generic")`**). Use this section to inspect quality when invoice and due dates are consistently present on that template.


In [13]:
T1_IMAGES_DIR = PROJECT_ROOT / "images"
t1_images = sorted(
    p
    for p in T1_IMAGES_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in SUPPORTED_EXTS
    and re.match(r"Template1_Instance\d+", p.stem, re.I)
)
print(f"Template 1 images: {len(t1_images)} files")
t1_images[:8]


Template 1 images: 200 files


[WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance0.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance1.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance10.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance100.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance101.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd semester/Statistical learning/group project/Document-Classifier/images/Template1_Instance102.jpg'),
 WindowsPath('c:/Users/rdjg2/OneDrive/Escritorio/ie/3/2nd

### Part C — Run extraction


In [14]:
results_t1 = []
errors_t1 = []

for idx, path in enumerate(t1_images, start=1):
    try:
        text = preprocess(path)
        row = extract_fields(text, layout="generic")
        row["template_id"] = 1
        row["file_name"] = path.name
        results_t1.append(row)
    except Exception as exc:
        errors_t1.append({"file_name": path.name, "error": str(exc)})
    if idx % 50 == 0:
        print(f"Part C: {idx}/{len(t1_images)}...")

print(f"Part C processed: {len(results_t1)}, errors: {len(errors_t1)}")
results_t1[:3]


Part C: 50/200...
Part C: 100/200...
Part C: 150/200...
Part C: 200/200...
Part C processed: 200, errors: 0


[{'invoice_number': '12345670',
  'invoice_date': '2008-03-20',
  'due_date': None,
  'issuer_name': 'Address:16424 Timothy Mission',
  'recipient_name': '16424 Timothy Mission',
  'total_amount': '734.33',
  'template_id': 1,
  'file_name': 'Template1_Instance0.jpg'},
 {'invoice_number': 'v-1999',
  'invoice_date': '1994-04-01',
  'due_date': None,
  'issuer_name': 'Address:16424 Timothy Mission',
  'recipient_name': 'Markville, AK 58294 US.',
  'total_amount': '92.46',
  'template_id': 1,
  'file_name': 'Template1_Instance1.jpg'},
 {'invoice_number': '12345670',
  'invoice_date': '2009-03-12',
  'due_date': None,
  'issuer_name': 'Address:16424 Timothy Mission',
  'recipient_name': '265 Ochoa Manors Apt. 518',
  'total_amount': '744.38',
  'template_id': 1,
  'file_name': 'Template1_Instance10.jpg'}]

### Part C — Save CSV


In [15]:
RESULTS_T1_CSV = PROJECT_ROOT / "images_template1_extraction_results.csv"
ERRORS_T1_CSV = PROJECT_ROOT / "images_template1_extraction_errors.csv"

t1_fields = [
    "template_id",
    "file_name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]
if results_t1:
    with RESULTS_T1_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=t1_fields, extrasaction="ignore")
        w.writeheader()
        for r in results_t1:
            w.writerow({k: r.get(k) for k in t1_fields})
    print("Saved:", RESULTS_T1_CSV)
else:
    print("No Part C rows to save.")

if errors_t1:
    with ERRORS_T1_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["file_name", "error"])
        w.writeheader()
        w.writerows(errors_t1)
    print("Saved:", ERRORS_T1_CSV)


Saved: c:\Users\rdjg2\OneDrive\Escritorio\ie\3\2nd semester\Statistical learning\group project\Document-Classifier\images_template1_extraction_results.csv


### Part C — Field coverage (missing vs filled)


In [16]:
def _field_missing_t1(v) -> bool:
    return v is None or (isinstance(v, str) and not v.strip())


req_t1 = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]
if results_t1:
    n = len(results_t1)
    for field in req_t1:
        miss = sum(1 for r in results_t1 if _field_missing_t1(r.get(field)))
        ok = n - miss
        print(f"- {field}: {ok} filled, {miss} missing (n={n})")
else:
    print("No Part C results.")


- invoice_number: 200 filled, 0 missing (n=200)
- invoice_date: 199 filled, 1 missing (n=200)
- due_date: 8 filled, 192 missing (n=200)
- issuer_name: 200 filled, 0 missing (n=200)
- recipient_name: 200 filled, 0 missing (n=200)
- total_amount: 200 filled, 0 missing (n=200)
